# 答えが資料にないときに「わかりません」と答えさせる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_rag/rag_refusal.ipynb)

NLP クックブック 20-06 ／ RAG (検索拡張生成)

検索結果に答えがないのに、LLM が無理に答えを作ってしまうのを防ぎたい。検索の段階と生成の段階の両方で歯止めをかける。

In [1]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

answerable = ["カレーはいくらですか？", "駐車場はありますか？", "月曜日は開いていますか？", "店主はどこで修行しましたか？"]
unanswerable = ["テイクアウトはできますか？", "アレルギー対応のメニューはありますか？", "明日の東京の天気は？", "日本で一番高い山は？"]
for label, qs in [("答えがある", answerable), ("答えがない", unanswerable)]:
    for q in qs:
        sims = doc_emb @ embedder.encode("query: " + q, normalize_embeddings=True)
        print(f"{label}  最大類似度 {sims.max():.3f}  {q}")

答えがある  最大類似度 0.825  カレーはいくらですか？
答えがある  最大類似度 0.850  駐車場はありますか？
答えがある  最大類似度 0.827  月曜日は開いていますか？
答えがある  最大類似度 0.814  店主はどこで修行しましたか？
答えがない  最大類似度 0.787  テイクアウトはできますか？
答えがない  最大類似度 0.801  アレルギー対応のメニューはありますか？
答えがない  最大類似度 0.768  明日の東京の天気は？
答えがない  最大類似度 0.715  日本で一番高い山は？


In [2]:
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=80):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def rag(question, threshold=0.80, k=2, strict=True):
    sims = doc_emb @ embedder.encode("query: " + question, normalize_embeddings=True)
    if sims.max() < threshold:                       # 1 段目: 関連する資料がそもそも見つからない
        return "(検索で棄却) 申し訳ありませんが、その質問にはお答えできません。"
    context = "\n".join(f"- {docs[i]}" for i in sims.argsort()[::-1][:k])
    rule = "資料に答えが書かれていない場合は、推測せずに「わかりません」とだけ答えてください。" if strict else ""
    prompt = f"以下の資料だけに基づいて、質問に一文で答えてください。{rule}\n\n# 資料\n{context}\n\n# 質問\n{question}"
    return chat([{"role": "user", "content": prompt}])   # 2 段目: 資料はあるが答えは載っていない場合は、LLM に断らせる

for q in ["カレーはいくらですか？", "明日の東京の天気は？", "テイクアウトはできますか？", "アレルギー対応のメニューはありますか？"]:
    print(f"Q: {q}\n   指示あり: {rag(q)}\n   指示なし: {rag(q, strict=False)}")

Q: カレーはいくらですか？
   指示あり: カレーは 950 円です。
   指示なし: カレーは看板メニューの特製ビーフカレーであれば 950 円です。
Q: 明日の東京の天気は？
   指示あり: (検索で棄却) 申し訳ありませんが、その質問にはお答えできません。
   指示なし: (検索で棄却) 申し訳ありませんが、その質問にはお答えできません。
Q: テイクアウトはできますか？
   指示あり: (検索で棄却) 申し訳ありませんが、その質問にはお答えできません。
   指示なし: (検索で棄却) 申し訳ありませんが、その質問にはお答えできません。
Q: アレルギー対応のメニューはありますか？
   指示あり: 資料に基づいて、アレルギー対応のメニューが提供されているかどうかは明記されていません。そのため、現在の情報だけでは判断できません。
   指示なし: 資料に基づくと、アレルギー対応のメニューが具体的に提供されているかは明記されていません。そのため、提供の有無について明確な回答はできません。しかし、一般的なレストランの対応として、事前に相談することでアレルギー対応の食事を提供してもらえる可能性はあります。より正確な情報を得るためには、直接店舗に問い合わせることをおすすめします。


- 類似度のしきい値だけでは「話題は近いが答えは載っていない」質問 (アレルギー対応) を弾けない。プロンプトでの指示と二段構えにする。
- 断る指示がなくても断れることはあるが、一般論を付け足して長くなりがち。指示があると短く言い切る。
- しきい値はモデルごとに違う (E5 は無関係な文でも 0.7 台後半が出る)。答えがある質問とない質問の例を用意して、分布を見て決める。
- 「わかりません」と言わせすぎると、答えられる質問まで断るようになる。断るべき質問と答えるべき質問の両方で評価する。